<a href="https://colab.research.google.com/github/Apur52027/depression_detection/blob/main/Optimized_Hybrid_Pipeline1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# =============================
# 📌 COMPLETE RUNNABLE NOTEBOOK
# Hybrid TF-IDF + BERT + Deep Learning
# =============================

# =============================
# 1. Install Dependencies
# =============================
!pip install transformers sentencepiece xgboost gensim -q


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 50.8 MB/s eta 0:00:00


In [ ]:
# =============================
# 2. Imports
# =============================
import pandas as pd
import numpy as np
import re
import nltk
import torch

from nltk.corpus import stopwords
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_selection import SelectKBest, chi2
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score
from scipy.sparse import hstack

from transformers import AutoTokenizer, AutoModel

In [ ]:
=============================
3. Load Dataset
=============================
df = pd.read_excel(
    'https://github.com/Apur52027/Dataset_ML/blob/main/BSMDD_v3_textcleaned%20-%2021K.xlsx?raw=true'
)

print("Initial shape:", df.shape)

# =============================

Initial shape: (21910, 2)


In [ ]:
# =============================
# 4. Cleaning
# =============================
df = df.drop_duplicates(subset=['text'])

df['word_count'] = df['text'].astype(str).apply(lambda x: len(x.split()))
df = df[df['word_count'] >= 30].drop(columns=['word_count'])

# Remove English chars
def remove_english(text):
    return re.sub(r'[^\u0980-\u09FF\s\?\!।]', '', str(text))

# Remove repeated punctuation
def clean_punct(text):
    text = re.sub(r'([\?\!।])\1+', r'\1', text)
    return re.sub(r'\s+', ' ', text).strip()

nltk.download('stopwords')
bangla_stopwords = set(stopwords.words('bengali'))

# Preprocess
def preprocess(text):
    text = remove_english(text)
    text = clean_punct(text)
    tokens = text.split()
    tokens = [w for w in tokens if w not in bangla_stopwords]
    return " ".join(tokens)


df['final_text'] = df['text'].apply(preprocess)

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


In [ ]:
# import pandas as pd

# data = {
#     "final_text": [
#         "আজকাল কিছুই ভালো লাগে না",
#         "জীবনটা খুব সুন্দর লাগছে",
#         "সবকিছু অর্থহীন মনে হচ্ছে",
#         "বন্ধুদের সাথে সময় কাটাতে ভালো লাগে",
#         "আমি খুব একা অনুভব করি",
#         "আজকে অনেক খুশি লাগছে",
#         "মনে হয় কেউ আমাকে বোঝে না",
#         "নতুন কিছু শিখতে ভালো লাগছে",
#         "ঘুম আসে না, সবসময় চিন্তা করি",
#         "পরিবারের সাথে সময় কাটানো দারুণ"
#     ],
#     "label": [
#         1,  # depressed
#         0,  # not depressed
#         1,
#         0,
#         1,
#         0,
#         1,
#         0,
#         1,
#         0
#     ]
# }

# df = pd.DataFrame(data)

# X = df['final_text']
# y = df['label']

# print(X)
# print(y)

In [ ]:
# =============================
# 5. Train-Test Split
# =============================
X = df['final_text']
y = df['label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)


In [ ]:
# =============================
# 6. TF-IDF
# =============================
max_f=3000
fs=2000
tfidf = TfidfVectorizer(max_features=max_f, ngram_range=(1,2), min_df=1, max_df=0.85) #max_feature=3000

X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

# Feature Selection
selector = SelectKBest(chi2, k=fs)
X_train_tfidf = selector.fit_transform(X_train_tfidf, y_train)
X_test_tfidf = selector.transform(X_test_tfidf)

In [ ]:
# =============================
# 7. BERT Embeddings
# =============================

tokenizer = AutoTokenizer.from_pretrained("distilbert-base-multilingual-cased")
model_bert = AutoModel.from_pretrained("distilbert-base-multilingual-cased")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_bert.to(device)


def get_embeddings(texts, batch_size=8):
    embeddings = []

    for i in range(0, len(texts), batch_size):
        batch = texts.iloc[i:i+batch_size].tolist()

        inputs = tokenizer(batch, return_tensors="pt", padding=True, truncation=True, max_length=64)
        inputs = {k: v.to(device) for k, v in inputs.items()}

        with torch.no_grad():
            outputs = model_bert(**inputs)

        cls = outputs.last_hidden_state[:, 0, :].cpu().numpy()
        embeddings.append(cls)

    return np.vstack(embeddings)


X_train_bert = get_embeddings(X_train)
X_test_bert = get_embeddings(X_test)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/466 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/542M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: distilbert-base-multilingual-cased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_transform.bias    | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
df.tail()

,text,label,final_text
21286,¸ – ¸á¦ “ á¦ “ ¸á¦ ” áêªšá¥´á • ¸êªáê ¸ á¥´êª ...,0,
21518,নিরামিষ শিক্ষক গ্রহের সবচেয়ে সুস মানুষদের নির...,0,নিরামিষ শিক্ষক গ্রহের সবচেয়ে সুস মানুষদের নির...
21539,সিরিয়াস ট্যাগ লাগাতে চাই পোস্টগুলো মুছে ®£ … ...,0,সিরিয়াস ট্যাগ লাগাতে চাই পোস্টগুলো মুছে
21576,কাম দানব ®£ ” ” ¸¨ • ” • ¥º — — ¯½ ¥±± ¹ª ½¯ •...,0,কাম দানব
21582,প্রতারক • — • ” ¹ª ¨ ± ± µ±¸ ª³³¹œ µ±± ±³ ¹µ ”...,0,প্রতারক


In [ ]:
# =============================
# 8. Hybrid Features
# =============================
X_train_final = hstack([X_train_tfidf, X_train_bert])
X_test_final = hstack([X_test_tfidf, X_test_bert])

In [ ]:
df.tail()

,text,label,final_text
21286,¸ – ¸á¦ “ á¦ “ ¸á¦ ” áêªšá¥´á • ¸êªáê ¸ á¥´êª ...,0,
21518,নিরামিষ শিক্ষক গ্রহের সবচেয়ে সুস মানুষদের নির...,0,নিরামিষ শিক্ষক গ্রহের সবচেয়ে সুস মানুষদের নির...
21539,সিরিয়াস ট্যাগ লাগাতে চাই পোস্টগুলো মুছে ®£ … ...,0,সিরিয়াস ট্যাগ লাগাতে চাই পোস্টগুলো মুছে
21576,কাম দানব ®£ ” ” ¸¨ • ” • ¥º — — ¯½ ¥±± ¹ª ½¯ •...,0,কাম দানব
21582,প্রতারক • — • ” ¹ª ¨ ± ± µ±¸ ª³³¹œ µ±± ±³ ¹µ ”...,0,প্রতারক


In [ ]:
# =============================
# 9. Logistic Regression
# =============================
model_lr = LogisticRegression(max_iter=1000)
model_lr.fit(X_train_final, y_train)

pred_lr = model_lr.predict(X_test_final)

print("\n===== Logistic Regression =====")
print(classification_report(y_test, pred_lr))



===== Logistic Regression =====
              precision    recall  f1-score   support

           0       0.87      0.89      0.88      2163
           1       0.85      0.82      0.84      1673

    accuracy                           0.86      3836
   macro avg       0.86      0.86      0.86      3836
weighted avg       0.86      0.86      0.86      3836



In [ ]:
# =============================
# 10. Other Models
# =============================
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

In [ ]:
svm_model = SVC(
    probability=True,
    random_state=42,
    kernel='rbf',
    C=1.0,
    gamma='scale'
)

In [ ]:
svm_model.fit(X_train_final, y_train)

SVC(probability=True, random_state=42)

In [ ]:
pred_lr = svm_model.predict(X_test_final)

print("\n\t\t====================SVM===================")
print(classification_report(y_test, pred_lr))



		====================SVM===================
              precision    recall  f1-score   support

           0       0.81      0.86      0.83      2163
           1       0.80      0.73      0.77      1673

    accuracy                           0.80      3836
   macro avg       0.80      0.80      0.80      3836
weighted avg       0.80      0.80      0.80      3836



In [ ]:
rf_model = RandomForestClassifier(
    n_estimators=200,        # more trees = better stability
    max_depth=None,          # full depth (or set like 10/20 to prevent overfitting)
    min_samples_split=2,
    min_samples_leaf=1,
    max_features='sqrt',     # best practice
    bootstrap=True,
    random_state=42,
    n_jobs=-1                # use all CPU cores
)

In [ ]:
rf_model.fit(X_train_final, y_train)

RandomForestClassifier(n_estimators=200, n_jobs=-1, random_state=42)

In [ ]:
pred_lr1 = rf_model.predict(X_test_final)

print("\n\t\t====================rf===================")
print(classification_report(y_test, pred_lr1))


		====================rf===================
              precision    recall  f1-score   support

           0       0.78      0.88      0.83      2163
           1       0.82      0.68      0.74      1673

    accuracy                           0.79      3836
   macro avg       0.80      0.78      0.79      3836
weighted avg       0.80      0.79      0.79      3836



In [ ]:
from xgboost import XGBClassifier

xgb_model = XGBClassifier(
    n_estimators=300,
    learning_rate=0.1,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric='logloss',
    random_state=42,
    n_jobs=-1
)

In [ ]:
xgb_model.fit(X_train_final, y_train)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.8, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='logloss',
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.1, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=6, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=300, n_jobs=-1,
              num_parallel_tree=None, ...)

In [ ]:
pred_lr = svm_model.predict(X_test_final)

print("\n\t\t====================xgb===================")
print(classification_report(y_test, pred_lr))


		====================xgb===================
              precision    recall  f1-score   support

           0       0.81      0.86      0.83      2163
           1       0.80      0.73      0.77      1673

    accuracy                           0.80      3836
   macro avg       0.80      0.80      0.80      3836
weighted avg       0.80      0.80      0.80      3836



In [ ]:
# =============================
# 11. Deep Learning (FastText + BiGRU)
# =============================
from gensim.models import FastText
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.layers import Embedding, Bidirectional, GRU, Dense, Dropout, Input
from tensorflow.keras.models import Model

# Train FastText
sentences = [text.split() for text in X_train]
fasttext_model = FastText(sentences, vector_size=100, window=5, min_count=2)

# Tokenizer
keras_tokenizer = Tokenizer(oov_token='<unk>')
keras_tokenizer.fit_on_texts(X_train)

word_index = keras_tokenizer.word_index

embedding_matrix = np.zeros((len(word_index)+1, 100))

for word, i in word_index.items():
    if word in fasttext_model.wv:
        embedding_matrix[i] = fasttext_model.wv[word]

# Sequences
X_train_seq = keras_tokenizer.texts_to_sequences(X_train)
X_test_seq = keras_tokenizer.texts_to_sequences(X_test)

X_train_pad = pad_sequences(X_train_seq, maxlen=100)
X_test_pad = pad_sequences(X_test_seq, maxlen=100)

# Model
inputs = Input(shape=(100,))

x = Embedding(len(word_index)+1, 100, weights=[embedding_matrix], trainable=False)(inputs)
x = Bidirectional(GRU(128, return_sequences=False))(x)
x = Dropout(0.5)(x)
x = Dense(64, activation='relu')(x)
outputs = Dense(1, activation='sigmoid')(x)

model_dl = Model(inputs, outputs)
model_dl.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])

model_dl.fit(X_train_pad, y_train, epochs=5, batch_size=16, validation_split=0.2)

# Evaluate
y_pred = (model_dl.predict(X_test_pad) > 0.5).astype(int)

print("\n===== Deep Learning =====")
print(classification_report(y_test, y_pred))

print("\n✅ DONE: Clean Hybrid Pipeline Working")

Epoch 1/5
768/768 ━━━━━━━━━━━━━━━━━━━━ 17s 15ms/step - accuracy: 0.8282 - loss: 0.3884 - val_accuracy: 0.8596 - val_loss: 0.3345
Epoch 2/5
768/768 ━━━━━━━━━━━━━━━━━━━━ 11s 15ms/step - accuracy: 0.8671 - loss: 0.3155 - val_accuracy: 0.8713 - val_loss: 0.2935
Epoch 3/5
768/768 ━━━━━━━━━━━━━━━━━━━━ 11s 15ms/step - accuracy: 0.8794 - loss: 0.2916 - val_accuracy: 0.8749 - val_loss: 0.2905
Epoch 4/5
768/768 ━━━━━━━━━━━━━━━━━━━━ 12s 15ms/step - accuracy: 0.8892 - loss: 0.2677 - val_accuracy: 0.8869 - val_loss: 0.2922
Epoch 5/5
768/768 ━━━━━━━━━━━━━━━━━━━━ 11s 15ms/step - accuracy: 0.9024 - loss: 0.2415 - val_accuracy: 0.8775 - val_loss: 0.3012
120/120 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step

===== Deep Learning =====
              precision    recall  f1-score   support

           0       0.90      0.88      0.89      2163
           1       0.85      0.87      0.86      1673

    accuracy                           0.88      3836
   macro avg       0.88      0.88      0.88      3836
weighted avg   

In [ ]:
from sklearn.ensemble import StackingClassifier

In [ ]:
estimators = [
    ('svm', svm_model),
    ('rf', rf_model),
    ('xgb', xgb_model)
]

stack_model = StackingClassifier(
    estimators=estimators,
    final_estimator=LogisticRegression(),
    cv=2 # Set cross-validation splits to 2
)

In [ ]:
stack_model.fit(X_train_final, y_train)

StackingClassifier(cv=2,
                   estimators=[('svm', SVC(probability=True, random_state=42)),
                               ('rf',
                                RandomForestClassifier(n_estimators=200,
                                                       n_jobs=-1,
                                                       random_state=42)),
                               ('xgb',
                                XGBClassifier(base_score=None, booster=None,
                                              callbacks=None,
                                              colsample_bylevel=None,
                                              colsample_bynode=None,
                                              colsample_bytree=0.8, device=None,
                                              early_stopping_rounds=None,
                                              enable_categorical=False...
                                              grow_policy=None,
                                              importance_type=None,
                                              interaction_constraints=None,
                                              learning_rate=0.1, max_bin=None,
                                              max_cat_threshold=None,
                                              max_cat_to_onehot=None,
                                              max_delta_step=None, max_depth=6,
                                              max_leaves=None,
                                              min_child_weight=None,
                                              missing=nan,
                                              monotone_constraints=None,
                                              multi_strategy=None,
                                              n_estimators=300, n_jobs=-1,
                                              num_parallel_tree=None, ...))],
                   final_estimator=LogisticRegression())

In [ ]:
pred_lr = stack_model.predict(X_test_final)

print("\n\t\t====================stack model===================")
print(classification_report(y_test, pred_lr))


		====================stack model===================
              precision    recall  f1-score   support

           0       0.85      0.88      0.86      2163
           1       0.84      0.80      0.82      1673

    accuracy                           0.85      3836
   macro avg       0.84      0.84      0.84      3836
weighted avg       0.85      0.85      0.85      3836

